In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score
import matplotlib.pyplot as plt
import warnings
import torch
from transformers import BertTokenizer, BertModel

# ----------------------------
# Mealpy algorithm imports
# ----------------------------
from mealpy.swarm_based import PSO, BA, CSO, FA, ABC
from mealpy.evolutionary_based import GA
from mealpy.swarm_based.ACOR import OriginalACOR
from mealpy.utils.problem import FloatVar
import xgboost as xgb

warnings.filterwarnings('ignore')

# ======================================================
# STEP 1: Load dataset + Use BERT for vectorization
# ======================================================
def load_data_with_bert(batch_size=8):
    # Load datasets
    amazon = pd.read_csv("../sentiment labelled sentences/amazon_cells_labelled.txt", sep="\t", header=None, names=["text", "label"])
    imdb   = pd.read_csv("../sentiment labelled sentences/imdb_labelled.txt", sep="\t", header=None, names=["text", "label"])
    yelp   = pd.read_csv("../sentiment labelled sentences/yelp_labelled.txt", sep="\t", header=None, names=["text", "label"])

    df = pd.concat([amazon, imdb, yelp], axis=0).reset_index(drop=True)
    
    # Load pre-trained BERT model and tokenizer
    tokenizer = BertTokenizer.from_pretrained('distilbert-base-uncased')
    model = BertModel.from_pretrained('distilbert-base-uncased')

    # Function to generate sentence embeddings using BERT
    def get_bert_embeddings(texts, batch_size=8):
        embeddings = []
        model.eval()
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i+batch_size]
                encoded = tokenizer(
                    batch, padding=True, truncation=True,
                    max_length=128, return_tensors='pt'
                )
                outputs = model(**encoded)
                # Mean pooling of last hidden states
                batch_embeddings = outputs.last_hidden_state.mean(dim=1)
                embeddings.append(batch_embeddings)
        return torch.cat(embeddings).numpy()

    print("🔄 Generating BERT embeddings... (This may take a few minutes)")
    X = get_bert_embeddings(df["text"].tolist(), batch_size=batch_size)
    y = df["label"].values

    print("✅ BERT embeddings generated successfully!")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test


# ======================================================
# STEP 2: Load Data
# ======================================================
X_train, X_test, y_train, y_test = load_data_with_bert(batch_size=8)


# ======================================================
# STEP 3: Objective Function for Multiple Models (Precision)
# ======================================================
def model_objective_function(solution, model_type):
    # Extract solution parameters
    max_depth         = int(solution[0])
    min_samples_split = int(solution[1])
    min_samples_leaf  = int(solution[2])
    criterion_idx     = int(solution[3])
    criterion         = 'gini' if criterion_idx == 0 else 'entropy'
    ccp_alpha         = float(solution[4])
    max_features      = float(solution[5])
    gamma_val         = float(solution[6])
    c_val             = float(solution[7])          # renamed to match PEP8
    learning_rate     = float(solution[8])          # added for GradientBoosting/XGBoost

    if model_type == "DecisionTree":
        model = DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth if max_depth > 0 else None,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            ccp_alpha=ccp_alpha,
            max_features=max_features,
            random_state=42
        )

    elif model_type == "RandomForest":
        model = RandomForestClassifier(
            criterion=criterion,
            max_depth=max_depth if max_depth > 0 else None,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42
        )

    elif model_type == "GradientBoosting":
        model = GradientBoostingClassifier(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            learning_rate=learning_rate,
            random_state=42
        )

    elif model_type == "KNN":
        model = KNeighborsClassifier(
            n_neighbors=max_depth,
            algorithm='auto'
        )

    elif model_type == "SVM":
        model = SVC(
            kernel='rbf',
            C=c_val,
            gamma=gamma_val,
            random_state=42
        )

    elif model_type == "XGBoost":
        model = xgb.XGBClassifier(
            max_depth=max_depth,
            min_child_weight=min_samples_split,
            gamma=min_samples_leaf,
            learning_rate=learning_rate,
            random_state=42
        )

    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        precision = precision_score(y_test, y_pred)
        return -precision
    except Exception as e:
        print(f"Error in {model_type}:", e)
        return 1.0


def define_problem(model_type):
    return {
        "bounds": [
            FloatVar(2, 30, "max_depth"),
            FloatVar(2, 10, "min_samples_split"),
            FloatVar(1, 10, "min_samples_leaf"),
            FloatVar(0, 1, "criterion"),
            FloatVar(0.0, 0.1, "ccp_alpha"),
            FloatVar(0.1, 1.0, "max_features"),
            FloatVar(0.001, 10.0, "gamma"),
            FloatVar(0.1, 10.0, "c_val"),
            FloatVar(0.01, 0.5, "learning_rate")   # Added learning rate
        ],
        "minmax": "min",
        "obj_func": lambda solution: model_objective_function(solution, model_type)
    }

# ======================================================
# STEP 5: Run All Metaheuristic Algorithms for Multiple Models
# ======================================================
algorithms = {
    "PSO": PSO.OriginalPSO(epoch=2, pop_size=5),
    "GA": GA.BaseGA(epoch=2, pop_size=5),
    "Bat": BA.OriginalBA(epoch=2, pop_size=5),
    "ACO": OriginalACOR(epoch=2, pop_size=5),
    "Cuckoo": CSO.OriginalCSO(epoch=2, pop_size=5),
    "Firefly": FA.OriginalFA(epoch=2, pop_size=5),
    "ABC": ABC.OriginalABC(epoch=2, pop_size=5)
}

models = ["DecisionTree", "RandomForest", "GradientBoosting", "KNN", "SVM", "XGBoost"]

results = {}

for model in models:
    print(f"\n🚀 Optimizing {model} using metaheuristic algorithms...")
    model_results = {}
    for name, optimizer in algorithms.items():
        print(f"\n  🚀 Running {name} for {model}...")
        problem = define_problem(model)
        best_agent = optimizer.solve(problem)
        best_solution = best_agent.solution
        best_fitness = -best_agent.target.fitness  # Convert back to positive precision
        
        best_params = {
            'max_depth': int(best_solution[0]),
            'min_samples_split': int(best_solution[1]),
            'min_samples_leaf': int(best_solution[2]),
            'criterion': 'gini' if int(best_solution[3]) == 0 else 'entropy'
        }
        
        model_results[name] = best_fitness
        print(f"✅ {name} Precision: {best_fitness:.4f}")
        print(f"🏆 {name} Best Params: {best_params}\n")
    
    results[model] = model_results


# ======================================================
# STEP 6: Visualization (Pie Chart)
# ======================================================
plt.figure(figsize=(15, 15))
for idx, (model, model_results) in enumerate(results.items()):
    plt.subplot(3, 2, idx+1)
    plt.pie(
        model_results.values(), labels=model_results.keys(),
        autopct='%1.1f%%', startangle=140,
        explode=[0.05]*len(model_results)
    )


c:\Users\ANIRUDDUHA\anaconda3\envs\tech\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'FloatVar' from 'mealpy.utils.problem' (c:\Users\ANIRUDDUHA\anaconda3\envs\tech\Lib\site-packages\mealpy\utils\problem.py)